In [1]:
### load xml yeast model with predefined directions '
import cobra
import pandas as pd
import sys
import os 
sys.path.append(os.path.abspath('../3_src/'))

import analysis.io_func as io_func
import helper_thermoflux.data_process as dp
import helper_thermoflux.build_yeast_model as bym

### compare reactions from excel model with yeast9 reactions, using EC number ?

In [2]:
dir_model=cobra.io.read_sbml_model(f'/home/nathan/PhD/04.Computational/202601_13C/1_data/c_synthetic_data/yeast-GEM9.xml')


Set parameter Username
Set parameter LicenseID to value 2749904
Set parameter GURO_PAR_DUMP to value 1
Read parameters from file gurobi.env
Academic license - for non-commercial use only - expires 2026-12-04


load tmodel to access smiles of metabolites/ accurate id of reactions


In [3]:
#need to load the lower and upper bound from this file 
df_dir_int=pd.read_excel('/home/nathan/Downloads/yeast_v3.2_starting_curated.xlsx',sheet_name='Reactions',index_col=0)
df_dir_transport=pd.read_excel('/home/nathan/Downloads/yeast_v3.2_starting_curated.xlsx',sheet_name='Transmembrane reactions',index_col=0)
df_tmodel_rxn=pd.concat([df_dir_int,df_dir_transport])
tmodel_ecs=df_tmodel_rxn['EC']

Unknown extension is not supported and will be removed
Conditional Formatting extension is not supported and will be removed
Unknown extension is not supported and will be removed
Conditional Formatting extension is not supported and will be removed


In [4]:
### now to find reactions in the Y9 model, make a dir of rid:ec-code (which can be alist of ec-codes)
y9_rxn_dict={}
for r in dir_model.reactions:
    try:
        y9_rxn_dict[r.id]=r.annotation['ec-code'] 
    except KeyError:
        pass
    

In [5]:
matches_id = {}
matches_rn = {}
mapping_id_rn={}
mapping_rn_id={}
for rxn, ec in y9_rxn_dict.items():
    ec_list = ec if isinstance(ec, list) else [ec]
    found = [e for e in ec_list if e in tmodel_ecs.values]
    if found:
        rid=tmodel_ecs[tmodel_ecs==found[0]].index.tolist()[0]
        matches_id[rid] = found
        matches_rn[rxn] = found
        mapping_id_rn[rxn]=rid
        mapping_rn_id[rid]=rxn



## complete with ORF matching rxn ?
i had matched/parsed our yeast model rxn id with ORF . notebook and resulting csv in 3/analysis

out put is  a df of common rxn that are also irreversibel 
and some manual identification

In [6]:
genes_mapping=pd.read_csv('/home/nathan/PhD/04.Computational/202601_13C/3_src/analysis/genes_mapping.csv',index_col=0)
genes_mapping

,ORF
Abbrevation,
ASNS1,"['YGR124W', 'YPR145W', 'YML096W']"
ASPK,['YER052C']
ASPTA,['YLR027C']
ASPTAm,['YKL106W']
ASAD,['YDR158W']
...,...
SUCFUMtm-2-2,['YJR095W']
THRtm+0,[None]
GALt+0,['YLR081W']


each y9 model reaction is annotated with corresponding genes. for each reaction 
- look into these genes (set)
-  go trhough the mapping df and check if any row (rxn) has a match with gene set
- make dict mapping Rnumber - RId 

In [7]:
dir(dir_model.reactions[0])#dir_model.reactions[0]#.gene_reaction_rule
dir_model.reactions[0].gpr.genes
orf_mapping_rn_rid={}
for r in dir_model.reactions:
    gene_set_r=r.gpr.genes
    for rid,row in genes_mapping.iterrows():
        if any([g in row['ORF'] for g in gene_set_r]):
            orf_mapping_rn_rid[r.id]=rid

## slow but whatever 

In [8]:
full_mapping_rn_rid=mapping_id_rn|orf_mapping_rn_rid
full_mapping_rn_rid

{'r_0007': 'PRMICIi',
 'r_0018': 'AATA',
 'r_0023': 'IPPMIa',
 'r_0027': 'HACNHm',
 'r_0029': 'ILETA',
 'r_0030': 'ILETA',
 'r_0032': 'BPNT',
 'r_0039': 'SHKK',
 'r_0040': 'SHKK',
 'r_0060': 'IPPMIa',
 'r_0061': 'IPMD',
 'r_0062': 'PYRDC',
 'r_0064': 'PYRDC',
 'r_0065': 'SHKK',
 'r_0069': 'THRS',
 'r_0080': 'MTHFR3',
 'r_0091': 'PGL',
 'r_0095': 'PYRDC',
 'r_0096': 'KARA2m',
 'r_0111': 'ACOAH',
 'r_0112': 'ACSm',
 'r_0113': 'ACSm',
 'r_0114': 'ACS',
 'r_0115': 'AGPRm',
 'r_0118': 'ACOTAm',
 'r_0148': 'ADK1',
 'r_0163': 'ALCD2x',
 'r_0165': 'ALCD2x',
 'r_0166': 'ALCD2x',
 'r_0167': 'ALCD2x',
 'r_0169': 'ALCD2x',
 'r_0170': 'ALCD2x',
 'r_0172': 'ALDD2xm',
 'r_0173': 'ALDD2y',
 'r_0177': 'ALDD2y',
 'r_0179': 'ALCD2x',
 'r_0180': 'ALCD2x',
 'r_0182': 'ALCD2x',
 'r_0183': 'ALCD2x',
 'r_0185': 'ALDD2xm',
 'r_0186': 'ALCD2x',
 'r_0187': 'ALCD2x',
 'r_0193': 'Trha',
 'r_0194': 'Trha',
 'r_0195': 'Tps2',
 'r_0198': 'MALTGD',
 'r_0202': 'ANPRT',
 'r_0203': 'IGPS',
 'r_0207': 'ARGSL',
 'r_0208': 

In [9]:
dict_irr_rxn={}
for rn in full_mapping_rn_rid.keys():
    if dir_model.reactions.get_by_id(rn).lower_bound==0 or dir_model.reactions.get_by_id(rn).upper_bound==0:
        dict_irr_rxn[full_mapping_rn_rid[rn]]=dir_model.reactions.get_by_id(rn).bounds


In [10]:
list_irr=[r.id for r in dir_model.reactions if r.lower_bound==0 or r.upper_bound==0]
len(list_irr)
df = pd.DataFrame(
    dict_irr_rxn
).T.set_axis(["lb", "ub"], axis=1)
# df.to_csv('yeast9_morecommon_irr_rxn.csv')

In [28]:
df=pd.DataFrame(dict_irr_rxn).T
df['rn']=[full_mapping_rid_rn[rid] for rid in dict_irr_rxn.keys()]
df.to_csv('y9_mapping_irr_auto.csv')

In [43]:
df.index[126:]

Index([], dtype='object')

In [11]:
full_mapping_rid_rn={v:k for k,v in full_mapping_rn_rid.items()}
for rid in ['ILETA', 'ACOAH', 'ALCD2x', 'ASPTAm', 'DAK', 'GCY1', 'TYRTA', 'VALTA',
       'THRA', 'SUCOAS1m', 'SULR', 'ORNtm+1-H', 'OAAtm-2', 'PItm-2', 'SO4t-2',
       'PHETA1']:
   print(rid,full_mapping_rid_rn[rid])
   print('y9',{met.name:stoich for met,stoich in dir_model.reactions.get_by_id(full_mapping_rid_rn[rid]).metabolites.items()})
   try : 
      print('tmodel',df_dir_int.loc[rid,'Reaction'])
   except:
      print('tmodel',df_dir_transport.loc[rid,'Reactant stoichiometry'])

wrong_dir_rxn=['ACOAH','ASPTAm','OAAtm-2','PItm-2','SO4t-2']

ILETA r_4355
y9 {'2-oxobutanoate': -1.0, 'L-glutamate': -1.0, '2-oxoglutarate': 1.0, '2-aminobutanoate': 1.0}
tmodel akg[c] + ile-L[c] <=> 3mop[c] + glu-L[c]
ACOAH r_4039
y9 {'acetate': -1.0, 'succinyl-CoA': -1.0, 'acetyl-CoA': 1.0, 'succinate': 1.0}
tmodel ac[c] + coa[c] <=> accoa[c] + h2o[c]
ALCD2x r_4671
y9 {'H+': -1.0, 'NADH': -1.0, '(4-hydroxyphenyl)acetaldehyde': -1.0, 'NAD': 1.0, 'tyrosol': 1.0}
tmodel etoh[c] + nad[c] <=> acald[c] + nadh[c]
ASPTAm r_0682
y9 {'2-oxoglutarate': -1.0, 'erythro-4-hydroxy-L-glutamic acid': -1.0, '4-hydroxy-2-oxoglutarate': 1.0, 'L-glutamate': 1.0}
tmodel akg[m] + asp-L[m] <=> glu-L[m] + oaa[m]
DAK r_4748
y9 {'ATP': -1.0, 'D-glyceraldehyde': -1.0, 'ADP': 1.0, 'glyceraldehyde 3-phosphate': 1.0, 'H+': 1.0}
tmodel dhap[c] + adp[c] <=> atp[c] + dha[c]
GCY1 r_0487
y9 {'glycerol': -1.0, 'NADP(+)': -1.0, 'glycerone': 1.0, 'H+': 1.0, 'NADPH': 1.0}
tmodel dha[c] + nadph[c] <=> glyc[c] + nadp[c]  
TYRTA r_1065
y9 {'3-(4-hydroxyphenyl)pyruvate': -1.0, 'L-glutam

In [29]:
#dump json full_mapping_rid_rn
import json
with open('full_mapping_rid_rn.json', 'w') as f:
    json.dump(full_mapping_rid_rn, f)

In [67]:
['r_1237','r_1574','r_3961','r_1266']
r=dir_model.reactions.get_by_id('r_1266')
for met in r.metabolites:
    print(met,met.name,met.compartment)
r

s_1468 sulphate e
s_1467 sulphate c


Reaction identifier,r_1266
Name,sulfate uniport
Memory address,0x7b8c28befaf0
Stoichiometry,s_1468 --> s_1467 sulphate --> sulphate
GPR,YBR294W or YLR092W
Lower bound,0.0
Upper bound,1000.0


In [33]:
'MALS' in df_dir_int.index

True

In [64]:
for r in dir_model.metabolites.get_by_id('s_1326').reactions:
    print(r.id,[m.name for m in r.metabolites])

r_1245 ['H+', 'phosphate', 'H+', 'phosphate']
r_0108 ['acetyl-CoA', 'ATP', 'bicarbonate', 'ADP', 'H+', 'malonyl-CoA', 'phosphate']
r_4337 ['ATP', 'H2O', 'porphyrin', 'ADP', 'H+', 'phosphate', 'porphyrin']
r_1022 ['ADP', 'phosphate', 'succinyl-CoA', 'ATP', 'coenzyme A', 'succinate']
r_0668 ['ATP', 'coenzyme A', 'itaconate', 'ADP', 'itaconyl-CoA', 'phosphate']
r_0226 ['ADP', 'H+', 'phosphate', 'ATP', 'H+', 'H2O']
r_4724 ['H2O', 'polyphosphate', 'H+', 'phosphate']
r_0569 ['diphosphate', 'H2O', 'H+', 'phosphate']
r_1264 ['phosphate', 'succinate', 'phosphate', 'succinate']
r_1623 ['5-formyltetrahydrofolic acid', 'ATP', '5,10-methenyl-THF', 'ADP', 'phosphate']
r_1226 ['(S)-malate', 'phosphate', '(S)-malate', 'phosphate']
r_4608 ['ATP', 'H2O', 'propionyl-CoA', 'ADP', 'H+', 'phosphate', 'propionyl-CoA']
r_4754 ['dimethylallyl phosphate', 'FMNH2', 'phosphate', 'prenylated FMNH2']
r_0447 ['ATP', 'formate', 'THF', '10-formyl-THF', 'ADP', 'phosphate']
r_3961 ['phosphate', 'phosphate']
r_0020 ['D-e

In [65]:
dir_model.reactions.get_by_id('r_1245')

Reaction identifier,r_1245
Name,phosphate transport
Memory address,0x7b8c28bedc30
Stoichiometry,s_0794 + s_1322 --> s_0799 + s_1326 H+ + phosphate --> H+ + phosphate
GPR,YER053C or YJR077C
Lower bound,0.0
Upper bound,1000.0
